# Day 5: Metadata and the Collector's Categories
**Date:** Thursday 25 June 2026

**Conceptual frame:** A corpus is organized by someone with a theory.
Analyzing the metadata is analyzing the collector's interpretive decisions.


```{admonition} Conceptual check — before you code
:class: tip

Answer the self-assessment questions for Day 5 before running the cells below.
Questions open in a new tab — come back here when you're done.

**[→ Open Day 5 Quiz](../quizpages/day5_quiz.md)**
```


---
## Part 0: Beregovski in Context

**Moshe Beregovski (1892–1961)** systematically documented Ukrainian Jewish music
in the 1930s and 40s. His work was suppressed under Stalin — he was arrested in 1950,
his manuscripts confiscated, and most of his research published only posthumously.

The metadata in this corpus reflects *his* categories, his collection choices,
and his informants' memories. When we find a pattern in the metadata, we must ask:
is this a feature of the music, the communities, or the collection methodology?

```{admonition} Recommended reading
Slobin, M. (1986). 'A Fresh Look at Beregovski's Folk Music Research.' *Ethnomusicology* 30(2).
```


In [ ]:
import requests,zipfile
from pathlib import Path
from collections import Counter
from itertools import islice
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from music21 import converter,note,interval
sns.set_theme(style='whitegrid',font_scale=1.1)
plt.rcParams['figure.figsize']=(10,4)
print('OK')

In [ ]:
CORPUS_DIR=Path('beregovski_corpus');KERN_DIR=CORPUS_DIR/'kern'
if not(KERN_DIR.exists() and list(KERN_DIR.glob('*.krn'))):
    CORPUS_DIR.mkdir(exist_ok=True)
    r=requests.get('https://github.com/shanahdt/mode_in_klezmer/archive/refs/heads/main.zip')
    zp=CORPUS_DIR/'repo.zip';zp.write_bytes(r.content)
    import shutil
    with zipfile.ZipFile(zp) as z:z.extractall(CORPUS_DIR)
    src=list(CORPUS_DIR.glob('mode_in_klezmer-*/kern'))
    if src:
        if KERN_DIR.exists():shutil.rmtree(KERN_DIR)
        shutil.copytree(src[0],KERN_DIR);zp.unlink()
print(f'{len(list(KERN_DIR.glob("*.krn")))} files')

In [ ]:
def load_corpus(kern_dir=KERN_DIR,verbose=True):
    pc2d={7:1,9:2,11:3,0:4,2:5,4:6,6:7,8:2,10:3,1:4,3:5,5:6}
    records,sdict={},{}
    for i,f in enumerate(sorted(Path(kern_dir).glob('*.krn'))):
        if verbose and i%50==0: print(f'  {i+1}...')
        try:
            s=converter.parse(str(f));ns=[n for n in s.flat.notes if isinstance(n,note.Note)]
            pcs=[n.pitch.pitchClass for n in ns]
            records[f.stem]={'tune_id':f.stem,'n_notes':len(ns),
                'pitches':[n.nameWithOctave for n in ns],'pitch_classes':pcs,
                'scale_degrees':[pc2d.get(p,0) for p in pcs],
                'intervals':[interval.Interval(ns[j],ns[j+1]).semitones for j in range(len(ns)-1)]}
            sdict[f.stem]=s
        except: pass
    if verbose: print(f'Loaded {len(records)}')
    return pd.DataFrame(records.values()),sdict

def get_ngrams(seq,n): return list(zip(*[islice(seq,i,None) for i in range(n)]))
print('ready')

In [ ]:
df,streams=load_corpus()
try:
    meta=pd.read_csv('https://raw.githubusercontent.com/shanahdt/mode_in_klezmer/main/metadata.csv')
    df=df.merge(meta,on='tune_id',how='left');print(f'{len(df)} tunes')
except Exception as e: print(e)

---
## Part 1: Exploring Metadata


In [ ]:
for col in ['mode','genre','region','source_type','instrument']:
    if col in df.columns:
        print(f'\n{col.upper()}:')
        print(df[col].value_counts().to_string())

In [ ]:
cats=[c for c in ['mode','genre','region'] if c in df.columns]
fig,axes=plt.subplots(1,len(cats),figsize=(14,4))
if len(cats)==1: axes=[axes]
for ax,col in zip(axes,cats):
    counts=df[col].value_counts()
    ax.barh(counts.index[:10],counts.values[:10],color='steelblue')
    ax.set_title(col);ax.set_xlabel('Count')
plt.tight_layout();plt.show()

---
## Part 2: Mode × Genre


In [ ]:
if 'mode' in df.columns and 'genre' in df.columns:
    ct=pd.crosstab(df['mode'],df['genre'])
    ct_n=pd.crosstab(df['mode'],df['genre'],normalize='index')
    fig,(ax1,ax2)=plt.subplots(1,2,figsize=(14,5))
    sns.heatmap(ct,annot=True,fmt='d',cmap='Blues',ax=ax1);ax1.set_title('Raw counts')
    sns.heatmap(ct_n,annot=True,fmt='.2f',cmap='YlOrRd',ax=ax2);ax2.set_title('Row proportions')
    plt.tight_layout();plt.show()
    chi2,p,dof,_=stats.chi2_contingency(ct)
    print(f'chi2={chi2:.2f}, p={p:.4f}, df={dof}')

---
## Part 3: Traveled Tunes


In [ ]:
if 'original_location' in df.columns and 'collection_location' in df.columns:
    df['traveled']=(df['original_location'].notna()&(df['original_location']!=df['collection_location']))
    print(f'Traveled: {df["traveled"].sum()}, Local: {(~df["traveled"]).sum()}')
    t,l=df[df.traveled]['n_notes'],df[~df.traveled]['n_notes']
    tstat,pval=stats.ttest_ind(t,l)
    print(f'Mean notes — traveled:{t.mean():.1f}, local:{l.mean():.1f}, t={tstat:.2f}, p={pval:.4f}')
else:
    print('Location columns not found in metadata')

---
## Day 5 Exercise: Chi-Square Interpretation

```{admonition} Exercise
Run a chi-square test on one metadata association of your choosing.
Write 200–250 words: does a significant association reflect the music,
the communities Beregovski documented, or his collection methodology?
```


In [ ]:
VAR_A='mode';VAR_B='genre'  # <-- change
if VAR_A in df.columns and VAR_B in df.columns:
    ct=pd.crosstab(df[VAR_A],df[VAR_B])
    print(ct)
    chi2,p,dof,_=stats.chi2_contingency(ct)
    print(f'chi2={chi2:.2f}, p={p:.4f}, df={dof}')

### Your interpretation

*(200–250 words)*


---
## Project Log — Entry 5

> *When I add metadata I find [X].*  
> *The most surprising result is [Y].*  
> *A question the metadata raises that melodic analysis cannot answer is...*

*(100–150 words)*


---
```{admonition} Weekend break — Fri 26 & Sat 27 June
Optional: listen to a commercial klezmer recording of a tune from your subset.
Write one paragraph: what does the recording convey that the kern score cannot?
Also a good time to set up your local Python environment if you haven't already.
```
